In [3]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from tqdm.auto import tqdm
from itertools import product

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '2_Propensities'))
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '4_Baselines' / '4.3_OutcomeModel'))
import MF_class as MF
import OM_class as OM

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

### 0. Choose Dataset

In [4]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Simulation'

# 1. Load Data

In [10]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Sequels'

train = pd.read_csv(data_path / 'train.csv')
test = pd.read_csv(data_path / 'test.csv')
with open(data_path / 'id2info.pkl', 'rb') as f:
    id2info = pickle.load(f)

train_users = train['user_id'].unique()
train_items = train['item_id'].unique()
test_users = test['user_id'].unique()

n_users = len(train_users)
n_items = len(train_items)

In [8]:
results = {'title_A': [], 'title_B': [], 'causal_link': []}
chosen_ids = []
for item1 in id2info:
    for item2 in id2info:
        if item1 == item2:
            continue
        if id2info[item1]['series'] != id2info[item2]['series']:
            continue
        num1 = id2info[item1]['number']
        num2 = id2info[item2]['number']
        if num1 == num2:
            continue
        if num1 < num2:
            link = 1
        else:
            link = 0
        results['title_A'].append(id2info[item1]['title'])
        results['title_B'].append(id2info[item2]['title'])
        results['causal_link'].append(link)
        chosen_ids.append((item1, item2))

oracle = pd.DataFrame(results)

# 2. Create Training and Validation Sets for Outcome Model

In [17]:
def get_timestamp_dict(data, users):

    subset = data[data['user_id'].isin(users)]
    pos = (
        subset[subset['interaction'] == 1]
        .groupby(['user_id', 'item_id'])['timestamp']
        .first()
    )

    timestamp_dict = {user_id : user_pos.droplevel(0).to_dict() for user_id, user_pos in tqdm(pos.groupby(level=0))}
    for user_id in users:
        if user_id not in timestamp_dict:
            timestamp_dict[user_id] = {}

    return timestamp_dict

def get_om_data(data, users, item_pairs_id, users_per_pair, random_state=42): 
    rng = np.random.default_rng(seed=random_state)
    len_users = len(users) 
    
    print("Constructing timestamp dictionary...")
    timestamp_dict = get_timestamp_dict(data, users)
    
    print("Generating OM training data...")
    rows = []
    for (i, j) in tqdm(item_pairs_id):
        for _ in range(users_per_pair):
            user = users[rng.integers(len_users)]
            pos_items_dict = timestamp_dict[user]

            x, y = 0, 0
            if i in pos_items_dict:
                if j in pos_items_dict:
                    if pos_items_dict[i] < pos_items_dict[j]:
                        x, y = 1, 1
                    else:
                        continue
                else:
                    x = 1
            elif j in pos_items_dict:
                y = 1
                    
            rows.append((user, i, j, x, y)) 
    
    return pd.DataFrame(rows, columns=["u", "i", "j", "x", "y"])

In [18]:
users_per_pair = 300

om_train_data = get_om_data(
    data=train, 
    users=train_users, 
    item_pairs_id=chosen_ids,
    users_per_pair=users_per_pair, 
    random_state=42)

om_test_data  = get_om_data(
    data=test,  
    users=test_users, 
    item_pairs_id=chosen_ids, 
    users_per_pair=users_per_pair, 
    random_state=42)

Constructing timestamp dictionary...


  0%|          | 0/7801 [00:00<?, ?it/s]

Generating OM training data...


  0%|          | 0/7340 [00:00<?, ?it/s]

Constructing timestamp dictionary...


  0%|          | 0/3877 [00:00<?, ?it/s]

Generating OM training data...


  0%|          | 0/7340 [00:00<?, ?it/s]

# 3. Save Datasets

In [20]:
om_train_data.to_csv(data_path / 'om_train.csv', index=False)
om_test_data.to_csv(data_path / 'om_test.csv', index=False)